# L07 · 왜 성공하거나 실패하는가

## Goal

**예상 시간:** 40분 · **경로:** full

- support overlap을 측정한다
- 실패 지표를 해석한다
- vOPD baseline을 설명한다

### 현재 위치: L06 → **L07** → L08

```text
Prompt/Data -> state source -> ... -> L07 -> ... -> fair evaluation
```

Alt text: The course map highlights L07 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L07"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L07', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

좋은 teacher가 항상 좋은 OPD를 만들지는 않는다. student가 방문한 state에서 top-k support가 겹치고, teacher가 새로운 유용한 신호를 주는지를 함께 본다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

OPD 실패는 최소 세 층으로 나눈다. (1) teacher 자체가 해당 state에서 틀림, (2) student와 teacher support가 너무 달라 유용한 token을 sample하지 못함, (3) estimator 분산·학습률·stale rollout 같은 최적화 문제다. loss 하나만으로 셋을 구분할 수 없다.

top-k overlap, entropy gap, forward/reverse KL, intervention rate를 함께 본다. vOPD는 top-k 기반 KL을 detached baseline으로 써 sampled estimator의 분산을 줄이려 한다. baseline은 기대 gradient를 바꾸지 않아야 하며 optimized target과 혼동하면 안 된다.

### 실제 구현: 왜 이렇게 만들었나

support 진단은 response mask에만 top-k 집합 overlap과 entropy gap을 계산한다. vOPD baseline은 detach되고 sampled-token log-prob만 gradient를 가진다. threshold는 성능 보장이 아니라 비교 run 간 경보 기준으로만 사용한다.

실제 코드: [`support.py`](../../src/opd_study/diagnostics/support.py), [`vopd.py`](../../src/opd_study/algorithms/vopd.py).

In [2]:
import inspect
from opd_study.diagnostics import support_diagnostics
from opd_study.algorithms import vopd_loss

objects_to_show = (support_diagnostics, vopd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.diagnostics.support.support_diagnostics
def support_diagnostics(
    student_logits: Tensor,
    teacher_logits: Tensor,
    mask: Tensor,
    *,
    top_k: int,
) -> SupportDiagnostics:
    """Measure top-k overlap and alignment only on explicitly selected states."""

    if student_logits.shape != teacher_logits.shape:
        raise ValueError("student and teacher logits must match")
    if mask.shape != student_logits.shape[:-1]:
        raise ValueError("mask must match non-vocabulary logits dimensions")
    vocabulary_size = student_logits.shape[-1]
    if not 1 <= top_k <= vocabulary_size:
        raise ValueError(f"top_k must be within [1, {vocabulary_size}]")
    selected_mask = mask.bool().reshape(-1)
    if not selected_mask.any().item():
        raise ValueError("support diagnostics require at least one selected token")
    student_log = torch.log_softmax(student_logits.float(), dim=-1).reshape(
        -1, vocabulary_size
    )[selected_mask]
    teacher_log = 

### 다른 선택지는 없나?

overlap을 높이는 처방은 teacher 교체, temperature 조정, off-policy/on-policy mixture, curriculum, multiple samples 등이다. 지표 하나가 나쁘다고 모두 적용하지 말고 실패 층을 먼저 분류한 뒤 한 변수씩 ablation한다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L07의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.diagnostics import support_diagnostics

student = torch.tensor([[[4., 3., 1., 0.], [0., 1., 3., 4.]]])
compatible = student + torch.tensor([[[.1, 0., 0., 0.], [0., 0., 0., .1]]])
incompatible = student.flip(-1)
mask = torch.tensor([[True, True]])
for name, teacher in (("compatible", compatible), ("incompatible", incompatible)):
    result = support_diagnostics(student, teacher, mask, top_k=2)
    print(name, {"overlap": result.overlap_ratio, "entropy_gap": result.absolute_entropy_gap})

compatible {'overlap': 1.0, 'entropy_gap': 0.029091089963912964}
incompatible {'overlap': 0.0, 'entropy_gap': 0.0}


In [4]:
from opd_study.algorithms import vopd_loss
from opd_study.data import CharacterTokenizer, collate_examples, generate_tiny_arithmetic
from opd_study.types import TeacherSignals

tokenizer = CharacterTokenizer(); row = generate_tiny_arithmetic(train_rows=1, validation_rows=1, test_rows=1).train
batch = collate_examples(row, tokenizer)
shape = (*batch.token_ids.shape, tokenizer.vocab_size)
student_logits = torch.randn(shape, requires_grad=True); teacher_logits = torch.randn(shape)
vopd = vopd_loss(student_logits, batch, TeacherSignals(logits=teacher_logits), baseline_top_k=8)
print("vOPD reward/advantage:", vopd.metrics["vopd/mean_reward"], vopd.metrics["vopd/mean_advantage"])
print("The top-k KL is a detached baseline, not the optimized target.")

vOPD reward/advantage: -0.010935024358332157 0.4987005889415741
The top-k KL is a detached baseline, not the optimized target.


## Checks

In [5]:
good = support_diagnostics(student, compatible, mask, top_k=2)
bad = support_diagnostics(student, incompatible, mask, top_k=2)
assert good.overlap_ratio > bad.overlap_ratio
assert vopd.loss.requires_grad
print("check passed: overlap diagnoses state compatibility; vOPD keeps sampled-token gradients")

check passed: overlap diagnoses state compatibility; vOPD keeps sampled-token gradients


**연습 (8분):** overlap은 낮지만 entropy gap은 작은 synthetic logits를 만들고, 이것만으로 teacher 품질 불량을 결론내릴 수 없는 이유를 쓰라.

<details><summary>확인 기준</summary>support 순위 불일치와 분포 sharpness는 다른 축이며 정답/환경 성공 근거가 추가로 필요하다.</details>

## 내가 자주 틀리는 것

### M1 — 낮은 overlap을 teacher 오답으로 단정하기

- 틀린 형태: top-k가 다르면 teacher가 나쁘다고 한다.
- 왜 틀렸나: student가 유용한 teacher mode를 아직 못 본 것일 수 있다.
- 고친 형태: correctness, entropy, KL과 environment 성공을 함께 본다.
- 관련 검사: `test_identical_support_is_perfectly_aligned`

### M2 — vOPD baseline에 gradient를 흘리기

- 틀린 형태: baseline까지 optimization target처럼 미분한다.
- 왜 틀렸나: score-function 기대 gradient가 바뀔 수 있다.
- 고친 형태: baseline을 detach하고 sampled log-prob만 미분한다.
- 관련 검사: `test_vopd_is_zero_when_teacher_equals_student`

## 60초 요약

1. support overlap을 측정한다
2. 실패 지표를 해석한다
3. vOPD baseline을 설명한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`rethinking_opd`](https://arxiv.org/abs/2604.13016v2) · `2604.13016v2` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`vopd`](https://arxiv.org/abs/2605.07865v1) · `2605.07865v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)